# How We Assign `U` to `X`

This notebook is generated from the uploaded document **“How we assign U → X.docx”**.

It explains how, when injecting a `U` node into an SF graph, the code decides which `X` node the `U` node should connect to.

## Core idea

The selection logic has three stages:

1. **Filter eligible `X` nodes**: keep only `X` nodes that can reach the target `Y` in the DAG.
2. **Sort eligible `X` nodes**: prioritize more connected / more influential nodes.
3. **Pick cyclically by `idx`**: select `reachable_x_nodes[idx % len(reachable_x_nodes)]`.

In [ ]:
# Key functions from the source logic

def _x_nodes_that_can_reach_y(graph, roles):
    """Return X nodes with a directed path to Y."""
    target_node = roles['Target_Node']
    return sorted(
        [node for node in roles['X_Nodes'] if nx.has_path(graph, node, target_node)],
        key=lambda node: (-graph.degree(node), -graph.out_degree(node), node)
    )


def _pick_reachable_x_nodes(reachable_x_nodes, start_index, count):
    if len(reachable_x_nodes) == 0:
        return []

    return [
        reachable_x_nodes[(start_index + offset) % len(reachable_x_nodes)]
        for offset in range(min(count, len(reachable_x_nodes)))
    ]

## Step 1: Filter eligible `X` nodes

The filter checks whether each `X` node has a directed path to the target node `Y`:

```python
nx.has_path(graph, node, target_node)
```

Only nodes satisfying the path condition below are kept:

```text
X -> ... -> Y
```

## Step 2: Sort the reachable `X` nodes

The sorting key is:

```python
key=lambda node: (-graph.degree(node), -graph.out_degree(node), node)
```

This means the priority order is:

1. **Larger total degree**: `graph.degree(node) = in_degree + out_degree`
2. **Larger out-degree**: prefer nodes influencing more downstream variables
3. **Smaller node index**: used as the final tie-breaker

In [ ]:
# Sorting demo with toy node statistics
import pandas as pd

x_stats = pd.DataFrame({
    'X_node': [4, 9, 7, 12],
    'degree': [5, 5, 3, 5],
    'out_degree': [3, 2, 2, 3],
})

sorted_x_stats = x_stats.sort_values(
    by=['degree', 'out_degree', 'X_node'],
    ascending=[False, False, True]
).reset_index(drop=True)

sorted_x_stats

## Step 3: Select by `idx` cyclically

For each `U` node, `idx` is the serial number inside its own role group.

For example, confounders and mediators each start counting from `0`.

In [ ]:
# Confounder idx example
confounder_nodes = [20, 21, 22]
pd.DataFrame({
    'confounder U': confounder_nodes,
    'idx': list(range(len(confounder_nodes)))
})

In [ ]:
# Mediator idx example: counting starts over from 0
mediator_nodes = [30, 31]
pd.DataFrame({
    'mediator U': mediator_nodes,
    'idx': list(range(len(mediator_nodes)))
})

The actual selection is equivalent to:

```python
x_child = reachable_x_nodes[idx % len(reachable_x_nodes)]
```

So the sorted list is reused in a cycle.

In [ ]:
# Cyclic selection demo
reachable_x_nodes = [4, 9, 7]
rows = []

for idx in range(5):
    selected_x = reachable_x_nodes[idx % len(reachable_x_nodes)]
    rows.append({'idx': idx, 'selected X': selected_x})

pd.DataFrame(rows)

## Final summary

The logic determining which `X` a `U` node connects to is:

1. Keep only `X` nodes that can reach `Y`: `X -> ... -> Y`
2. Sort them by:
   - larger total degree
   - larger out-degree
   - smaller node index
3. Select cyclically with:

```python
reachable_x_nodes[idx % len(reachable_x_nodes)]
```